In [9]:
select top 5  *
from ecommerce_raw;

(5 rows affected)

order_id         | order_date       | ship_date    | delivery_date    | customer_id | customer_name     | customer_email                | customer_segment | sales_channel | country        | region | product_id  | product_name              | category | subcategory | quantity | unit_price | unit_cogs | discount_pct | gross_revenue | net_revenue | cogs_total | gross_profit | gross_margin_pct | shipping_cost | return_flag | return_reason | return_date | refund_amount | inventory_on_hand | reorder_point | days_in_stock | supplier_id | supplier_lead_time_days | payment_method | payment_status | fulfillment_status | order_status | warehouse_id | notes         
-----------------+------------------+--------------+------------------+-------------+-------------------+-------------------------------+------------------+---------------+----------------+--------+-------------+---------------------------+----------+-------------+----------+------------+-----------+--------------+---

In [20]:
Select category, 
        subcategory,
        sum(round(net_revenue,0)) as Revenue,
        sum(round(gross_profit,0)) as Profit, 
        round(AVG(gross_margin_pct),2) AS Margin, 
        count(order_id) AS total_orders,
        (sum(net_revenue) * 100) / SUM(sum(net_revenue)) OVER() As Revenue_percentage

from FactSales
LEFT JOIN DimProduct on FactSales.product_id = DimProduct.product_id
GROUP BY category,subcategory
order by Revenue desc

(15 rows affected)

category | subcategory | Revenue  | Profit  | Margin   | total_orders | Revenue_percentage
---------+-------------+----------+---------+----------+--------------+-------------------
Office   | Desks       | 10003592 | 5284225 | 0.550000 | 7197         | 10.104012045259553
office   | Accessories | 8881881  | 4214574 | 0.470000 | 7041         | 8.971113699455987 
Bedroom  | Bedding     | 8193839  | 4624626 | 0.570000 | 7212         | 8.276277399049258 
Storage  | Boxes       | 7246421  | 4009181 | 0.510000 | 7192         | 7.319020779542074 
Kitchen  | Cookware    | 7200424  | 4266302 | 0.600000 | 7135         | 7.27272070259734  
Kitchen  | Cutlery     | 7146890  | 4025545 | 0.540000 | 7298         | 7.218571420257797 
Kitchen  | Utensils    | 6974451  | 3699697 | 0.550000 | 6047         | 7.04521165993137  
Storage  | Shelves     | 6852506  | 3689289 | 0.540000 | 5920         | 6.92102751123534  
Outdoor  | Patio       | 6546161  | 3897589 | 0.590000 | 5913         

Office leads revenue and profit, but kitchen and outdoor leads margin by 56% & 58% 

Office Accessories has the worst margin in the entire dataset at 47%, dragging the category average down. Office Desks at 55% and Chairs at 58% are healthy, but Accessories is quietly eating profit. Office leads revenue by a wide margin yet sits at 53% category margin precisely because of this subcategory. That's a concrete, actionable finding.

Cookware (60%), Cutlery (54%), Utensils (55%) — the tightest margin band of any category. No subcategory is underperforming. This is why Kitchen is the real star despite not leading revenue.

Bedding sits at 57%, Decor at 57%, but Lighting drops to 48%. Nearly 10 percentage points below its siblings. That's not a small gap — something structural is going on there, either pricing, COGS, or heavy discounting.

Boxes (51%) vs Shelves (54%) vs Baskets (54%) — margins are fine, but Baskets only generates $2.9M vs Boxes at $7.2M. Storage is heavily concentrated in Boxes. If that subcategory has a supply or demand issue, the whole category feels it.

Same margin as Patio (58-59%) but only $3.1M revenue vs $6.5M for Patio and $6M for Lighting. Less than half the orders of the other subcategories. Either it's a niche product line or it's underexposed in the catalog.



In [33]:
SELECT
    category,
    subcategory,
    ROUND(AVG(discount_pct), 3) AS avg_discount,
    ROUND(AVG(gross_margin_pct), 2) AS avg_margin,
     COUNT(FactReturns.order_id) * 100
        / COUNT(FactSales.order_id) AS return_rate_pct
FROM FactSales
LEFT JOIN DimProduct ON FactSales.product_id = DimProduct.product_id
LEFT JOIN FactReturns ON FactSales.order_id = FactReturns.order_id
GROUP BY category, subcategory
ORDER BY avg_discount DESC

(15 rows affected)

category | subcategory | avg_discount | avg_margin | return_rate_pct
---------+-------------+--------------+------------+----------------
Office   | Accessories | 0.078000     | 0.470000   | 26             
Outdoor  | Patio       | 0.078000     | 0.590000   | 28             
Office   | Desks       | 0.078000     | 0.550000   | 27             
Office   | Chairs      | 0.078000     | 0.580000   | 27             
Outdoor  | Gardening   | 0.077000     | 0.580000   | 26             
Storage  | Shelves     | 0.076000     | 0.540000   | 26             
Bedroom  | Lighting    | 0.076000     | 0.480000   | 27             
Bedroom  | Decor       | 0.076000     | 0.570000   | 27             
Kitchen  | Cutlery     | 0.076000     | 0.540000   | 27             
Storage  | Baskets     | 0.075000     | 0.540000   | 27             
Storage  | Boxes       | 0.075000     | 0.510000   | 33             
Kitchen  | Utensils    | 0.075000     | 0.550000   | 27             
Outdoor  | Lig

discounting is consistent across categories, so margin variation is not driven by promotional pricing
* Storage Boxes at 33% return rate but everything else is clustering between 26–28% return rate


In [29]:
SELECT count(FactSales.order_id),count(FactReturns.order_id)
from FactSales
LEFT JOIN  FactReturns on FactSales.order_id = FactReturns.order_id

(1 row affected)

(No column name) | (No column name)
-----------------+-----------------
102205           | 28556           
(1 row)

Total execution time: 00:00:02.116

In [37]:
SELECT  Year(order_date) AS Year,sum(round(net_revenue,0)) as Revenue,
        sum(round(gross_profit,0)) as Profit, 
        round(AVG(gross_margin_pct),2) AS Margin, 
        count(order_id) AS total_orders
FROM FactSales
GROUP BY Year(order_date)


(5 rows affected)

Year | Revenue  | Profit   | Margin   | total_orders
-----+----------+----------+----------+-------------
2022 | 32209138 | 17652075 | 0.550000 | 32904       
2023 | 33359556 | 18228346 | 0.550000 | 32915       
2024 | 32916805 | 18094543 | 0.550000 | 32744       
2025 | 81       | 39       | 0.480000 | 1           
2027 | 519340   | 278346   | 0.550000 | 489         
(5 rows)

Total execution time: 00:00:02.379

In [43]:
SELECT top 20 order_date,ship_date, FactSales.order_id,return_date
FROM FactSales
LEFT JOIN FactReturns on FactSales.order_id = FactReturns.order_id
Where YEAR(order_date) = 2027 ;

(20 rows affected)

order_date | ship_date  | order_id         | return_date
-----------+------------+------------------+------------
2027-03-18 | 2024-11-08 | ORD-329764-50396 | NULL       
2027-01-22 | 2023-12-11 | ORD-466063-30266 | NULL       
2027-03-31 | 2024-01-20 | ORD-477427-28754 | NULL       
2027-03-29 | 2023-06-27 | ORD-549037-64747 | NULL       
2027-01-04 | 2023-06-19 | ORD-829736-31444 | 2023-07-08 
2027-04-06 | 2022-04-25 | ORD-904098-54994 | NULL       
2027-02-22 | 2022-12-30 | ORD-934381-58032 | NULL       
2027-03-11 | 2023-02-04 | ORD-944576-21203 | NULL       
2027-02-11 | 2024-09-24 | ORD-433232-34526 | NULL       
2027-01-05 | 2022-10-04 | ORD-535804-62636 | NULL       
2027-02-21 | 2023-10-22 | ORD-593348-18075 | NULL       
2027-03-13 | 2022-06-19 | ORD-694707-38501 | NULL       
2027-01-07 | 2024-09-02 | ORD-106092-14376 | NULL       
2027-02-13 | 2023-05-26 | ORD-189870-13511 | NULL       
2027-01-12 | 2024-01-05 | ORD-268049-27868 | NULL       
2027-02-07 

In [ ]:
SELECT order_date,ship_date,return_date,
        CASE 
          WHEN YEAR(ship_date) < YEAR(order_date) 
                THEN REPLACE(order_date,  YEAR(order_date),YEAR(ship_date))
          
          else Null
          end as order_date
from FactSales
LEFT JOIN FactReturns ON FactSales.order_id = FactReturns.order_id
Where YEAR(order_date) = 2027; 


In [ ]:
SELECT COALESCE(
    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 101)) < YEAR(TRY_CONVERT(DATETIME, order_date, 101))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 101) AS DATE),
    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 101)) < YEAR(TRY_CONVERT(DATETIME, order_date, 101))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 101) AS DATE),
    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 101)) < YEAR(TRY_CONVERT(DATETIME, order_date, 105))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 105) AS DATE),

    TRY_CAST(TRY_CONVERT(DATETIME,
        CASE 
            WHEN YEAR(TRY_CONVERT(DATETIME, ship_date, 101)) < YEAR(TRY_CONVERT(DATETIME, order_date, 120))
            THEN REPLACE(order_date, CAST(YEAR(order_date) AS VARCHAR), CAST(YEAR(ship_date) AS VARCHAR))
            ELSE order_date
        END
    , 12O)  AS DATE)
) AS order_date


FROM ecommerce_raw


Msg 102, Level 15, State 1, Line 30
Incorrect syntax near 'O'.

Total execution time: 00:00:00.167